<a href="https://colab.research.google.com/github/maliaheupel/ds2002-fa2/blob/main/04-%20Json/2026_10_09_%E2%80%94_Public_API_to_Clean_Table_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [17]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}


def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    last = None
    # TODO: loop up to `tries` times
    for attempt in range(tries):
    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
      try:
        r = requests.get(BASE, params=params, timeout=10)
        if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
    # TODO:   raise on a status that is not in RETRY_STATUSES
        if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(
                    f'not retryable: {r.status_code} {r.text[:200]}')
        last = f'status {r.status_code}'
      except requests.RequestException as e:
        last = repr(e)
    # TODO:   otherwise sleep (increasing each time) and try again
      time.sleep(1 + attempt)
    # TODO: raise if every attempt failed
    raise RuntimeError(f"every attempt failed after {tries} tries")


### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [19]:
# TODO: call get_weather once for a range that spans both dates,
try:
  weather = get_weather('2026-09-05', '2026-10-03')
#       then keep just the two dates you care about
  weather = weather[weather['time'].isin(['2026-09-05', '2026-10-03'])]
except Exception as e:
    print(f"API call failed: {e}")
used_fallback = False

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [20]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'
if used_fallback:
    wx = FALLBACK.copy()
    wx['source'] = 'fallback sample'
    used_fallback = True
else:
    wx = weather.copy()
    wx['source'] = 'open-meteo'

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [22]:
# TODO
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm',
})
wx['date'] = pd.to_datetime(wx['date'])
wx = wx[['date', 'temp_max', 'precip_mm', 'source']]

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [23]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [25]:
# TODO
wettest = wx.loc[wx['precip_mm'].idxmax()]
driest = wx.loc[wx['precip_mm'].idxmin()]

diff = wettest['precip_mm'] - driest['precip_mm']
print(f"{wettest['date'].strftime('%Y-%m-%d')} was wetter by {diff:.1f} mm.")

2026-10-03 was wetter by 12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [29]:
# TODO
print(f"Data Source Citation:")
print(f"Endpoint: {BASE}")
print(f"Coordinates: (38.03, -78.51)")
print(f"Date Range: 2026-09-05 to 2026-10-03")
print(f"Pulled At: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"ata Status: {'fallback sample' if used_fallback else 'live Open-Meteo API'}")

Data Source Citation:
Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: (38.03, -78.51)
Date Range: 2026-09-05 to 2026-10-03
Pulled At: 2026-10-08 20:45:16
ata Status: live Open-Meteo API


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

When it gets merged, every row will be filled with NaN for weather. When charting it against revenue, visualization libraries drop the NaN rows, leaving the plot blank or showing zero correlation. Nobody would catch this because the code would still run without any error messages and enter the final analysis.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [30]:
# stretch
# city1 - charlottesville
wx['city'] = 'Charlottesville'

# orlando
orlando_raw = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38)
orlando_wx = orlando_raw[orlando_raw['time'].isin(['2026-09-05', '2026-10-03'])].copy()

orlando_wx = orlando_wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})
orlando_wx['date'] = pd.to_datetime(orlando_wx['date'])
orlando_wx['source'] = 'open-meteo'
orlando_wx['city'] = 'Orlando'

# combine into 1
combined = pd.concat([wx, orlando_wx], ignore_index=True)
combined

,date,temp_max,precip_mm,source,city
0,2026-09-05,29.4,2.5,open-meteo,Charlottesville
1,2026-10-03,22.0,14.7,open-meteo,Charlottesville
2,2026-09-05,32.1,3.9,open-meteo,Orlando
3,2026-10-03,30.4,14.5,open-meteo,Orlando
